# TransE, TransH y TransR, paso a paso

Solo estos tres modelos y solo el grafo de SteelVault, el más pequeño y el único con las partes
verificadas a mano. Cinco pasos:

1. el grafo como tripletas `(h, r, t)`;
2. qué hace cada modelo;
3. entrenarlos y ver qué han aprendido;
4. ¿adivinan aristas que no han visto?;
5. la pregunta que nos importa: ¿adivinan la parte de un enunciado?, y por qué no.

Los modelos y su configuración son los mismos que en
[`link_prediction.ipynb`](link_prediction.ipynb), así que los números del paso 5 coinciden con
los de allí. Corre en la GPU; debería tardar un par de minutos.

In [ ]:
import collections
import json
import time

import numpy as np
import torch
from IPython.display import Markdown, display

import link_prediction as lp

SV = lp.load_contract("SteelVault")
e2i, r2i = lp.entity_ids(SV)  # cada entidad y cada tipo de arista recibe un número
i2e = {i: e for e, i in e2i.items()}
MODELS = ("TransE", "TransH", "TransR")
best = json.loads((lp.RESULTS_DIR / "results.json").read_text())["ajuste:SteelVault"]["result"]["best"]
CONFIG = {m: {k: v for k, v in best[m].items() if k != "MRR validación"} for m in MODELS}
print("dispositivo:", lp.DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

## 1. El grafo como tripletas

Cada arista del grafo es una tripleta `(h, r, t)`: un origen, un tipo de arista y un destino.

In [ ]:
print(f"{len(SV.triples)} tripletas, {len(e2i)} entidades, {len(r2i)} tipos de arista presentes")
lp.table([{"tipo de arista": r, "tripletas": n} for r, n in collections.Counter(r for _, r, _ in SV.triples).most_common()])

En SteelVault no aparecen `depends_on` ni `supersedes`: la extracción no produjo ninguna.

Algunas tripletas, con el texto de cada nodo para poder leerlas:

In [ ]:
def label(node):
    if node in SV.parties:
        return f"{node} ({SV.parties[node]['name']})"
    if node in SV.clauses:
        c = SV.clauses[node]
        return f"{node} (§{c['ref']} {c.get('heading') or ''})".strip()
    if node in SV.statements:
        return f"{node} «{lp.statement_text(SV.statements[node])[:55]}…»"
    return node


examples = [("obligation-3", "assigns_obligation_to", "party-2"), ("obligation-3", "is_part_of", "clause-1"),
            ("right-3", "grants_right_to", "party-1"), ("condition-3", "is_part_of", "obligation-7"),
            ("right-15", "modifies", "clause-22")]
assert all(t in set(SV.triples) for t in examples)
lp.table([{"h": label(h), "r": r, "t": label(t)} for h, r, t in examples])

## 2. Qué hace cada modelo

Los tres dan a cada entidad un vector de números (64 o 128, según la configuración) y a cada
tipo de arista otro vector. Una tripleta es «verdadera» si sus vectores encajan de una cierta
forma; cada modelo define esa forma:

| Modelo | Una tripleta es verdadera si… | Puntuación (más alta = más creíble) |
|---|---|---|
| **TransE** | origen + relación ≈ destino | `−‖h + r − t‖` |
| **TransH** | lo mismo, pero después de proyectar origen y destino sobre un **plano propio de cada relación** | `−‖h⊥ + r − t⊥‖`, con `h⊥ = h − (wᵣ·h) wᵣ` |
| **TransR** | lo mismo, pero después de llevar origen y destino a **un espacio propio de cada relación** con una matriz | `−‖Mᵣh + r − Mᵣt‖` |

**El problema del muchos-a-uno**, con los datos de SteelVault: 40 aristas `assigns_obligation_to`
apuntan a la misma parte, `party-2`. TransE quiere que, para las 40, `h + r ≈ party-2`, así que
empuja a los 40 enunciados hacia el mismo punto, `party-2 − r`. Pero esos enunciados también
tienen que estar en sitios distintos para sus otras aristas (cada uno en su cláusula). TransH y
TransR nacieron para esto: solo exigen que coincidan **en la proyección** de esa relación, y fuera
de ella cada enunciado puede estar donde le haga falta.

## 3. Entrenar y ver qué han aprendido

**Entrenar** es mover los vectores para que las tripletas verdaderas puntúen más que las falsas.
Para cada tripleta conocida se le pregunta al modelo `(h, r, ?)`: se puntúan las 183 entidades
como posible destino, y se corrigen los vectores para que la verdadera quede arriba. Lo mismo
con el origen, `(?, r, t)`. Se repite unas cien o trescientas veces sobre todas las tripletas
(las *épocas*).

Primero, con **todas** las tripletas. ¿Cuánto tarda en la GPU?

In [ ]:
models = {}
rows = []
for m in MODELS:
    start = time.time()
    models[m] = lp.train_kge(m, SV.triples, e2i, r2i, seed=0, **CONFIG[m])
    rows.append({"modelo": m, **CONFIG[m], "segundos": round(time.time() - start, 1)})
lp.table(rows, ["modelo", "epochs", "embedding_dim", "lr", "loss", "segundos"])

Segundos por modelo. Con un grafo de 275 tripletas, entrenar no es lo lento.

**Cómo se lee un modelo.** Se le hace una pregunta, por ejemplo `(obligation-3,
assigns_obligation_to, ?)` —¿a quién obliga *«Marketing Affiliate shall pay to Equidata…»*?—. El
modelo puntúa las 183 entidades y las ordena. La posición de la respuesta correcta es su
**rango**: 1 si sale la primera.

Las cinco primeras de cada modelo. La respuesta correcta es `party-2`, y el modelo la ha visto al
entrenar: esto mide memoria, no deducción.

In [ ]:
def top(model, h, r, k=5):
    scores = lp.kge_tail_scores(model, [(h, r)], e2i, r2i)[0]
    order = np.argsort(-scores)
    return [f"{i2e[i]} ({scores[i]:.2f})" for i in order[:k]]


lp.table([{"modelo": m, **{f"{k + 1}.º": x for k, x in enumerate(top(models[m], "obligation-3", "assigns_obligation_to"))}} for m in MODELS])

Los números entre paréntesis son las puntuaciones. Solo importa el orden, no su valor:
cada modelo usa su propia escala.

## 4. ¿Adivinan aristas que no han visto?

Ahora la prueba de verdad: se esconde un 10% de las tripletas al azar (27), se entrena con el
resto y se pregunta por las escondidas. Para resumir 27 rangos en un número se usan dos medidas:

- **MRR** (*rango recíproco medio*): la media de `1 / rango`. Si la respuesta sale 1.ª, 2.ª y
  4.ª en tres preguntas, el MRR es `(1 + 1/2 + 1/4) / 3 = 0,58`. Vale 1 si siempre sale la primera.
- **Hits@k**: la fracción de preguntas en que la respuesta sale entre las *k* primeras.

El rango es **filtrado**: si una pregunta tiene varias respuestas verdaderas conocidas (una
cláusula tiene muchos enunciados), las otras no cuentan como error. Como referencia, el azar:
ordenar las 183 entidades sin mirar nada.

In [ ]:
rng = np.random.default_rng(0)
hidden_idx = set(rng.choice(len(SV.triples), size=len(SV.triples) // 10, replace=False).tolist())
train = [t for i, t in enumerate(SV.triples) if i not in hidden_idx]
hidden = [SV.triples[i] for i in sorted(hidden_idx)]
true_tails = collections.defaultdict(set)
for h, r, t in SV.triples:
    true_tails[(h, r)].add(e2i[t])


def filtered_ranks(model, triples):
    scores = lp.kge_tail_scores(model, [(h, r) for h, r, _ in triples], e2i, r2i)
    ranks = []
    for row, (h, r, t) in zip(scores, triples):
        row = row.copy()
        row[[i for i in true_tails[(h, r)] if i != e2i[t]]] = -np.inf
        ranks.append(1 + int(np.sum(row > row[e2i[t]])))
    return np.array(ranks)


n = len(e2i)
rows = [{"modelo": "al azar (esperado)", "MRR": float(np.mean([1 / k for k in range(1, n + 1)])), "Hits@1": 1 / n, "Hits@10": 10 / n}]
ranks_by_model = {}
for m in MODELS:
    ranks = filtered_ranks(lp.train_kge(m, train, e2i, r2i, seed=0, **CONFIG[m]), hidden)
    ranks_by_model[m] = ranks
    rows.append({"modelo": m, "MRR": float(np.mean(1 / ranks)), "Hits@1": float(np.mean(ranks == 1)), "Hits@10": float(np.mean(ranks <= 10))})
display(Markdown(f"{len(hidden)} tripletas escondidas"))
lp.table(rows)

Como referencia, en el ajuste de [`link_prediction.ipynb`](link_prediction.ipynb) —otra muestra
del 10% y dos semillas— el MRR fue 0,46 para TransE, 0,23 para TransH y 0,11 para TransR.

El rango de cada tripleta escondida, para ver en qué tipo de arista acierta cada modelo:

In [ ]:
lp.table([{"h": h, "r": r, "t": t, **{m: int(ranks_by_model[m][k]) for m in MODELS}} for k, (h, r, t) in enumerate(hidden)])

Cómo leerla: un 1 es un acierto a la primera; un número alto, que la respuesta quedó muy abajo
entre las 183 entidades. Fíjate en qué tipos de arista (`r`) dan rangos bajos y en cuáles altos, y
en si el origen `h` tiene otras aristas que el modelo haya podido ver.

TransH y TransR tienen más piezas que ajustar que TransE (un plano o una matriz por tipo de
arista) y los mismos datos: en un grafo con casi todos los nodos de una a tres aristas, les
faltan ejemplos para ajustarlas. No es un fallo del entrenamiento: con el mismo código, en
*Nations* —un grafo de prueba pequeño pero denso— TransH empata con TransE
([`link_prediction.ipynb`](link_prediction.ipynb), sección 3).

## 5. La pregunta que nos importa: ¿qué parte?

Se esconde la arista de parte de un enunciado y se le pregunta al modelo cuál de las **dos**
partes encaja mejor. Con dos candidatas, el MRR no aporta nada que no diga el acierto (vale
`(1 + acierto) / 2`), así que se mide el **acierto**: la fracción de enunciados con la parte bien.

- Solo los **47 enunciados unilaterales**. Los 36 gemelos —la misma frase emitida una vez por
  parte, como en *«each party shall…»*— tienen las dos partes por construcción y no hay nada que
  adivinar.
- Se reparten los 47 en cinco grupos; se esconde un grupo, se entrena con el resto del grafo y se
  pregunta por ese grupo. Así hasta pasar por los cinco, y todo tres veces con repartos distintos.
- Dos referencias sin modelo: **siempre la parte más frecuente** y **la mayoría de su cláusula**
  (la parte que más aparece entre los otros enunciados de la misma cláusula).

In [ ]:
queries = [q for q in lp.party_queries(SV) if q[0] not in SV.twin]
twins = [q for q in lp.party_queries(SV) if q[0] in SV.twin]
parties = sorted(SV.parties)
correct = collections.defaultdict(list)  # método -> [acierto de cada repetición]
predictions = collections.defaultdict(dict)  # método -> {enunciado: parte} de la primera repetición
start = time.time()
for rep in range(3):
    fold_of = lp.folds(len(queries), 5, seed=rep)
    hits = collections.Counter()
    for fold in range(5):
        test = [q for q, f in zip(queries, fold_of) if f == fold]
        known = [q for q, f in zip(queries, fold_of) if f != fold] + twins
        graph = [t for t in SV.triples if t not in set(test)]
        base = lp.t1_baselines(SV, known, test)
        guesses = {"siempre la más frecuente": base["mayoría"], "mayoría de su cláusula": base["cláusula"]}
        for m in MODELS:
            scores = lp.kge_tail_scores(lp.train_kge(m, graph, e2i, r2i, seed=rep, **CONFIG[m]), [(h, r) for h, r, _ in test], e2i, r2i)
            guesses[m] = {sid: parties[int(np.argmax([row[e2i[p]] for p in parties]))] for (sid, _, _), row in zip(test, scores)}
        for method, guess in guesses.items():
            hits[method] += sum(guess[sid] == p for sid, _, p in test)
            if rep == 0:
                predictions[method].update(guess)
    for method, k in hits.items():
        correct[method].append(k / len(queries))
print(f"{3 * 5 * len(MODELS)} entrenamientos en {time.time() - start:.0f} s")
lp.table(sorted([{"método": m, "acierto": float(np.mean(v)), "peor repetición": min(v), "mejor repetición": max(v)} for m, v in correct.items()], key=lambda r: -r["acierto"]))

En [`link_prediction.ipynb`](link_prediction.ipynb), con estos mismos modelos y repartos, ninguno
de los tres llegó a la mayoría de su cláusula, ni siquiera a «siempre la más frecuente», y TransH y
TransR, pensados para el muchos-a-uno, quedaron por debajo de TransE. Esta tabla debería repetirlo.

### Por qué

Al esconder la arista de parte, ¿qué le queda al modelo para decidir? Las otras aristas del
enunciado. Cuántas tiene cada uno de los 47:

In [ ]:
others = collections.Counter(
    sum(1 for h, r, t in SV.triples if sid in (h, t) and r not in lp.PARTY_RELATIONS) for sid, _, _ in queries
)
lp.table([{"aristas además de la de parte": k, "enunciados": others[k]} for k in sorted(others)])

La mayoría tiene una o dos: casi siempre `is_part_of` hacia su cláusula y, a veces, un término
definido que usa o una condición que cuelga de él. **Ninguna de esas aristas dice quién carga con
el enunciado.** Lo único que el modelo puede deducir es «se parece a los otros enunciados de su
cláusula», que es justo la mayoría de la cláusula, y eso hecho peor. Los que no tienen ninguna
otra arista quedan aislados: el modelo no tiene nada que mirar.

El muchos-a-uno era un problema real para TransE, pero no el que importa aquí. TransH y TransR lo
resuelven dejando que cada enunciado tenga su sitio lejos de la parte, y con eso tampoco hay de
dónde sacar la parte.

Lo que dice quién carga está **en el texto**, no en el grafo. Unos casos en que TransE falla:

In [ ]:
fails = [(sid, p) for sid, _, p in queries if predictions["TransE"][sid] != p][:6]
lp.table([{
    "enunciado": sid,
    "texto": lp.statement_text(SV.statements[sid])[:80],
    "lo que ve el modelo": ", ".join(f"{r} {t if h == sid else h}" for h, r, t in SV.triples if sid in (h, t) and r not in lp.PARTY_RELATIONS) or "nada",
    "TransE dice": SV.parties[predictions["TransE"][sid]]["name"][:22],
    "correcta": SV.parties[p]["name"][:22],
} for sid, p in fails])

## En resumen

- El grafo de SteelVault son 275 tripletas `(h, r, t)` entre 183 entidades.
- Los tres modelos aprenden algo del grafo en general (paso 4), TransE más que los otros dos.
- Para la parte de un enunciado (paso 5) no superan a mirar la mayoría de su cláusula. No es por
  el modelo: el grafo no contiene la información. Cada enunciado tiene una o dos aristas aparte
  de la de su parte, y ninguna dice quién carga con él.
- Corre en la GPU, y entrenar cada modelo lleva segundos.